In [1]:
# --- Imports / path setup (match your script) ---
import sys
from pathlib import Path
import numpy as np

sys.path.insert(0, str(Path.cwd().parent))

from src.behavior_import.import_data import *
from src.behavior_import.extract_trials import *
from src.behavior_analysis.get_total_reversals import *
from src.behavior_analysis.get_good_reversal_info import *
from src.behavior_analysis.get_choice_probs_around_good_reversals import *
from src.behavior_analysis.split_good_reversals_by_best_change import *
from src.behavior_analysis.split_early_late_good_reversals import *
from src.behavior_visualization.plot_choice_probs_around_good_reversals import *

In [4]:
task = "open-field"
cohort = "cohort-01"
folder_name = "3x3_field_blocked_reward_bandit"

root = f"/Volumes/behrens/meg/{folder_name}/{cohort}/rawdata/"
subjects_data = import_data(root)
subjects_trials_by_problem = extract_trials_grouped_by_problem(subjects_data)

[WARN] Failed to read /Volumes/behrens/meg/3x3_field_blocked_reward_bandit/cohort-01/rawdata/sub-01_id-MY_71_N/ses-08_date-20260204/behav/._MY_71_N-2026-02-04-153602.tsv: 'utf-8' codec can't decode byte 0xb0 in position 37: invalid start byte.
[WARN] Failed to read /Volumes/behrens/meg/3x3_field_blocked_reward_bandit/cohort-01/rawdata/sub-05_id-MY_72_L/ses-155_date-20260613/behav/._MY_72_L-2026-06-13-201430.tsv: 'utf-8' codec can't decode byte 0xb0 in position 37: invalid start byte.
[WARN] Failed to read /Volumes/behrens/meg/3x3_field_blocked_reward_bandit/cohort-01/rawdata/sub-02_id-MY_71_L/ses-54_date-20260305/behav/._MY_71_L-2026-03-05-150416.tsv: 'utf-8' codec can't decode byte 0xb0 in position 37: invalid start byte.
[WARN] Failed to read /Volumes/behrens/meg/3x3_field_blocked_reward_bandit/cohort-01/rawdata/sub-04_id-MY_72_N/ses-40_date-20260225/behav/._MY_72_N-2026-02-25-143929.tsv: 'utf-8' codec can't decode byte 0xb0 in position 37: invalid start byte.
[WARN] Failed to read /

In [5]:
ex_sess = subjects_data["MY_71_R"]["ses-91_date-20260330"]

In [9]:
print(ex_sess.keys())

dict_keys(['data', 'df', 'problem', 'choice_towers', 'initiation_tower', 'has_good', 'has_bad', 'trial_variables', 'trial_info', 'trial', 'blocks', 'trials_in_block', 'trials_since_last_reversal', 'choice', 'chosen_rank', 'num_long_pokes', 'long_pokes', 'reward_pulses', 'reward_magnitudes', 'total_reward_amount', 'best_arm', 'choice_counts', 'rank_counts', 'ema_best_arm_choices', 'ema_second_arm_choices', 'can_switch', 'num_trials_until_switch', 'total_num_trials_until_switch', 'reward_magnitudes_by_tower', 'chosen_magnitude', 'reward_magnitude_by_arm', 'reward_magnitude_by_rank', 'choices_by_tower', 'choices_by_rank'])


In [15]:
import json
import pandas as pd
import numpy as np

keys_to_save = [
    'trial',
    'blocks',
    'trials_in_block',
    'choice',
    'chosen_rank',
    'num_long_pokes',
    'long_pokes',
    'reward_pulses',
    'reward_magnitudes',
    'total_reward_amount',
    'choice_counts',
    'rank_counts',
    'reward_magnitudes_by_tower',
    'chosen_magnitude',
    'reward_magnitude_by_arm',
    'reward_magnitude_by_rank',
    'choices_by_tower',
    'choices_by_rank'
]

def is_trial_list(x):
    return isinstance(x, (list, tuple, np.ndarray, pd.Series))

def make_json_safe(x):
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, (np.integer, np.floating)):
        return x.item()
    return x

# Use trial length as number of rows
n_trials = len(ex_sess["trial"])

data = {}

for k in keys_to_save:
    v = ex_sess[k]

    if is_trial_list(v) and len(v) == n_trials:
        # trial-by-trial column
        data[k] = [
            json.dumps(make_json_safe(x)) if isinstance(x, (dict, list, tuple, np.ndarray)) else x
            for x in v
        ]
    else:
        # session-level dict/list/scalar repeated on every row
        data[k] = [json.dumps(make_json_safe(v))] * n_trials

df = pd.DataFrame(data)

df.head()

,trial,blocks,trials_in_block,choice,chosen_rank,num_long_pokes,long_pokes,reward_pulses,reward_magnitudes,total_reward_amount,choice_counts,rank_counts,reward_magnitudes_by_tower,chosen_magnitude,reward_magnitude_by_arm,reward_magnitude_by_rank,choices_by_tower,choices_by_rank
0,1,1,6,A1,second,1,"[""B3""]",1,"{""B1"": 0, ""A1"": 1, ""C1"": 2}",2,"{""B1"": 0, ""A1"": 1, ""C1"": 0}","{""best"": 0, ""second"": 1, ""third"": 0}","{""A1"": [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,...",1,"{""A1"": [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,...","{""best"": [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...","{""A1"": [1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1,...","{""best"": [0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, ..."
1,2,1,7,A1,second,4,"[""C2"", ""C3"", ""B3"", ""A2""]",1,"{""B1"": 0, ""A1"": 1, ""C1"": 2}",4,"{""B1"": 0, ""A1"": 2, ""C1"": 0}","{""best"": 0, ""second"": 2, ""third"": 0}","{""A1"": [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,...",1,"{""A1"": [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,...","{""best"": [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...","{""A1"": [1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1,...","{""best"": [0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, ..."
2,3,1,8,C1,best,0,[],2,"{""B1"": 0, ""A1"": 1, ""C1"": 2}",8,"{""B1"": 0, ""A1"": 2, ""C1"": 1}","{""best"": 1, ""second"": 2, ""third"": 0}","{""A1"": [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,...",2,"{""A1"": [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,...","{""best"": [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...","{""A1"": [1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1,...","{""best"": [0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, ..."
3,4,1,9,B1,third,0,[],0,"{""B1"": 0, ""A1"": 1, ""C1"": 2}",8,"{""B1"": 1, ""A1"": 2, ""C1"": 1}","{""best"": 1, ""second"": 2, ""third"": 1}","{""A1"": [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,...",0,"{""A1"": [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,...","{""best"": [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...","{""A1"": [1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1,...","{""best"": [0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, ..."
4,5,1,10,C1,best,0,[],2,"{""B1"": 0, ""A1"": 1, ""C1"": 2}",12,"{""B1"": 1, ""A1"": 2, ""C1"": 2}","{""best"": 2, ""second"": 2, ""third"": 1}","{""A1"": [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,...",2,"{""A1"": [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,...","{""best"": [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...","{""A1"": [1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1,...","{""best"": [0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, ..."


In [16]:
df.to_csv("session.csv", index=False)

In [11]:
import pandas as pd

session = ex_sess['trial_info']
df = pd.DataFrame(session)

In [12]:
df.head()

,0,1,2,3,4,5,6,7,8,9,...,99,100,101,102,103,104,105,106,107,108
0,"{'num_t': 1, 'num_t_in_block': 6, 'num_blocks'...","{'num_t': 2, 'num_t_in_block': 7, 'num_blocks'...","{'num_t': 3, 'num_t_in_block': 8, 'num_blocks'...","{'num_t': 4, 'num_t_in_block': 9, 'num_blocks'...","{'num_t': 5, 'num_t_in_block': 10, 'num_blocks...","{'num_t': 6, 'num_t_in_block': 11, 'num_blocks...","{'num_t': 7, 'num_t_in_block': 12, 'num_blocks...","{'num_t': 8, 'num_t_in_block': 13, 'num_blocks...","{'num_t': 9, 'num_t_in_block': 14, 'num_blocks...","{'num_t': 10, 'num_t_in_block': 15, 'num_block...",...,"{'num_t': 100, 'num_t_in_block': 4, 'num_block...","{'num_t': 101, 'num_t_in_block': 5, 'num_block...","{'num_t': 102, 'num_t_in_block': 6, 'num_block...","{'num_t': 103, 'num_t_in_block': 7, 'num_block...","{'num_t': 104, 'num_t_in_block': 8, 'num_block...","{'num_t': 105, 'num_t_in_block': 9, 'num_block...","{'num_t': 106, 'num_t_in_block': 10, 'num_bloc...","{'num_t': 107, 'num_t_in_block': 11, 'num_bloc...","{'num_t': 108, 'num_t_in_block': 12, 'num_bloc...","{'num_t': 109, 'num_t_in_block': 13, 'num_bloc..."


In [ ]:
df.to_csv("session.csv", index=False)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from src.behavior_import.import_data import *
from src.behavior_import.extract_trials import *
from src.behavior_analysis.get_good_reversal_info import *
from src.behavior_analysis.get_rank_counts_by_good_reversal import *
from src.behavior_analysis.get_diagnostic_p_value import *
from src.behavior_visualization.plot_rank_proportions import *
sys.path.append(str(Path(__file__).resolve().parent.parent))
from fix_grid_maze_cohort_02_problems import *

# task = "grid-maze"
task = "open-field"

folder_name = None
cohort = None
if task == "grid-maze":
    cohort = "cohort-02"
    folder_name = "3x3_maze_blocked_reward_bandit"
elif task == "open-field":
    cohort = "cohort-02"
    folder_name = "3x3_field_blocked_reward_bandit"
root = f"/Volumes/behrens/meg/{folder_name}/{cohort}/rawdata/"

subjects_data = import_data(root)
subjects_trials_by_problem = extract_trials_grouped_by_problem(subjects_data)
if task == "grid-maze" and cohort == "cohort-02":
    subjects_trials_by_problem = fix_grid_maze_cohort_02_problems(subjects_trials_by_problem)

for problem_number in subjects_trials_by_problem.keys():
    print(problem_number)
    problem = f"problem-{problem_number:02d}"
    
    subjects_trials = subjects_trials_by_problem[problem_number]

    reversal_windows = get_good_reversal_info(subjects_trials, include_first_block=True)
    rank_counts_by_good_reversal = get_rank_counts_by_good_reversal(reversal_windows)
    p_values = pvalue_paired_t_best_vs_second_vs_third(rank_counts_by_good_reversal)

    save_path = f"../../results/figures/{task}/{cohort}/{problem}/choice-stats/Rank Proportions"
    plot_rank_proportions(rank_counts_by_good_reversal, average_across_mice_pvalues=p_values, save_path=save_path)